In [0]:
bronze_path = "abfss://bronze@stretaildataeng26.dfs.core.windows.net/online_retail.csv"

df_bronze = (
    spark.read
    .option("header", "true")
    .csv(bronze_path)
)

display(df_bronze.limit(5))

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850,United Kingdom
536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850,United Kingdom


In [0]:
print("Total records:", df_bronze.count())

print("Total columns:", len(df_bronze.columns))

df_bronze.printSchema()

Total records: 541909
Total columns: 8
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Country: string (nullable = true)



In [0]:
from pyspark.sql.functions import col, to_timestamp
df_typed = (
    df_bronze
    .withColumn("Quantity", col("Quantity").cast("int"))
    .withColumn("UnitPrice", col("UnitPrice").cast("double"))
    .withColumn("InvoiceDate", to_timestamp(col("InvoiceDate"), "MM/d/yy H:mm"))
)

df_typed.printSchema()
display(df_typed.limit(5))

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Country: string (nullable = true)



InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01T08:26:00Z,2.55,17850,United Kingdom
536365,71053,WHITE METAL LANTERN,6,2010-12-01T08:26:00Z,3.39,17850,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01T08:26:00Z,2.75,17850,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01T08:26:00Z,3.39,17850,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01T08:26:00Z,3.39,17850,United Kingdom


In [0]:
from pyspark.sql.functions import col, coalesce, try_to_timestamp, lit

df_typed = (
    df_bronze
    .withColumn("Quantity", col("Quantity").cast("int"))
    .withColumn("UnitPrice", col("UnitPrice").cast("double"))
    .withColumn(
        "InvoiceDate",
        coalesce(
            try_to_timestamp(col("InvoiceDate"), lit("M/d/yyyy H:mm")),
            try_to_timestamp(col("InvoiceDate"), lit("M/d/yy H:mm"))
        )
    )
)

print("Records:", df_typed.count())
df_typed.printSchema()

Records: 541909
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Country: string (nullable = true)



In [0]:
from pyspark.sql.functions import (
    col, coalesce, try_to_timestamp, lit
)

df_typed = (
    df_bronze
    .withColumn("Quantity", col("Quantity").cast("int"))
    .withColumn("UnitPrice", col("UnitPrice").cast("double"))
    .withColumn(
        "InvoiceDate",
        coalesce(
            try_to_timestamp(col("InvoiceDate"), lit("M/d/yyyy H:mm")),
            try_to_timestamp(col("InvoiceDate"), lit("M/d/yy H:mm"))
        )
    )
)

df_typed.printSchema()
display(df_typed.limit(5))

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Country: string (nullable = true)



InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01T08:26:00Z,2.55,17850,United Kingdom
536365,71053,WHITE METAL LANTERN,6,2010-12-01T08:26:00Z,3.39,17850,United Kingdom
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01T08:26:00Z,2.75,17850,United Kingdom
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01T08:26:00Z,3.39,17850,United Kingdom
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01T08:26:00Z,3.39,17850,United Kingdom


In [0]:
from pyspark.sql.functions import col

print(
    "Negative quantities:",
    df_typed.filter(col("Quantity") < 0).count()
)

print(
    "Zero quantities:",
    df_typed.filter(col("Quantity") == 0).count()
)

print(
    "Negative prices:",
    df_typed.filter(col("UnitPrice") < 0).count()
)

print(
    "Zero prices:",
    df_typed.filter(col("UnitPrice") == 0).count()
)

Negative quantities: 10624
Zero quantities: 0
Negative prices: 2
Zero prices: 2515


In [0]:
from pyspark.sql.functions import col

display(
    df_typed
    .filter(col("Quantity") < 0)
    .select(
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "UnitPrice"
    )
    .limit(10)
)

InvoiceNo,StockCode,Description,Quantity,UnitPrice
C536379,D,Discount,-1,27.5
C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,4.65
C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,1.65
C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,0.29
C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,0.29
C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,0.29
C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,3.45
C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,1.65
C536391,22553,PLASTERS IN TIN SKULLS,-24,1.65
C536506,22960,JAM MAKING SET WITH JARS,-6,4.25


In [0]:
from pyspark.sql.functions import col, when

df_silver = df_typed.withColumn(
    "TransactionStatus",
    when(
        col("InvoiceNo").startswith("C"),
        "Cancelled"
    ).otherwise("Completed")
)

display(
    df_silver.select(
        "InvoiceNo",
        "Quantity",
        "UnitPrice",
        "TransactionStatus"
    ).limit(10)
)

InvoiceNo,Quantity,UnitPrice,TransactionStatus
536365,6,2.55,Completed
536365,6,3.39,Completed
536365,8,2.75,Completed
536365,6,3.39,Completed
536365,6,3.39,Completed
536365,2,7.65,Completed
536365,6,4.25,Completed
536366,6,1.85,Completed
536366,6,1.85,Completed
536367,32,1.69,Completed


In [0]:
# Count records by transaction status
status_counts = (
    df_silver
    .groupBy("TransactionStatus")
    .count()
)

display(status_counts)

TransactionStatus,count
Completed,532621
Cancelled,9288


In [0]:
from pyspark.sql.functions import col, round

df_silver = df_silver.withColumn(
    "LineTotal",
    round(
        col("Quantity") * col("UnitPrice"),
        2
    )
)

display(
    df_silver.select(
        "InvoiceNo",
        "Quantity",
        "UnitPrice",
        "LineTotal",
        "TransactionStatus"
    ).limit(10)
)

InvoiceNo,Quantity,UnitPrice,LineTotal,TransactionStatus
536365,6,2.55,15.3,Completed
536365,6,3.39,20.34,Completed
536365,8,2.75,22.0,Completed
536365,6,3.39,20.34,Completed
536365,6,3.39,20.34,Completed
536365,2,7.65,15.3,Completed
536365,6,4.25,25.5,Completed
536366,6,1.85,11.1,Completed
536366,6,1.85,11.1,Completed
536367,32,1.69,54.08,Completed


In [0]:
# Count total records
total_records = df_silver.count()

# Count distinct records
unique_records = df_silver.dropDuplicates().count()

# Calculate duplicates
duplicate_records = total_records - unique_records

print("Total records:", total_records)
print("Unique records:", unique_records)
print("Duplicate records:", duplicate_records)

Total records: 541909
Unique records: 536641
Duplicate records: 5268


In [0]:
# Remove exact duplicate rows from the Silver DataFrame
df_silver_clean = df_silver.dropDuplicates()

# Verify the result
print("Silver records:", df_silver_clean.count())
print("Silver columns:", len(df_silver_clean.columns))

display(df_silver_clean.limit(5))

Silver records: 536641
Silver columns: 10


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TransactionStatus,LineTotal
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01T08:26:00Z,2.55,17850,United Kingdom,Completed,15.3
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01T08:26:00Z,3.39,17850,United Kingdom,Completed,20.34
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01T08:26:00Z,3.39,17850,United Kingdom,Completed,20.34
536365,71053,WHITE METAL LANTERN,6,2010-12-01T08:26:00Z,3.39,17850,United Kingdom,Completed,20.34
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01T08:26:00Z,2.75,17850,United Kingdom,Completed,22.0


In [0]:
# Silver Delta Lake storage path
silver_path = (
    "abfss://silver@stretaildataeng26.dfs.core.windows.net/"
    "retail_transactions"
)

# Save cleaned Silver data in Delta format
(
    df_silver_clean.write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)

print("Silver Delta data saved successfully!")

Silver Delta data saved successfully!


In [0]:
# Read the saved Silver Delta dataset
silver_path = (
    "abfss://silver@stretaildataeng26.dfs.core.windows.net/"
    "retail_transactions"
)

df_silver_verified = (
    spark.read
    .format("delta")
    .load(silver_path)
)

print("Silver records:", df_silver_verified.count())
print("Silver columns:", len(df_silver_verified.columns))

display(df_silver_verified.limit(5))

Silver records: 536641
Silver columns: 10


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TransactionStatus,LineTotal
536373,37370,RETRO COFFEE MUGS ASSORTED,6,2010-12-01T09:02:00Z,1.06,17850,United Kingdom,Completed,6.36
536425,22508,DOORSTOP RETROSPOT HEART,8,2010-12-01T12:08:00Z,3.75,13758,United Kingdom,Completed,30.0
536464,22952,60 CAKE CASES VINTAGE CHRISTMAS,2,2010-12-01T12:23:00Z,0.55,17968,United Kingdom,Completed,1.1
536477,22595,CHRISTMAS GINGHAM HEART,144,2010-12-01T12:27:00Z,0.72,16210,United Kingdom,Completed,103.68
536537,21694,SMALL REGAL SILVER CANDLEPOT,12,2010-12-01T13:51:00Z,2.95,15922,United Kingdom,Completed,35.4
